# Vision-language navigation with InternVLA-N1 DualVLN and OpenVINO

[InternVLA-N1](https://github.com/InternRobotics/InternNav) is a vision-language navigation (VLN)
foundation model. Given a natural-language route description such as *"Go straight to the hallway and
then turn left. Go past the bed..."* and the robot's camera stream, it decides how the robot should
move next. The [DualVLN](https://arxiv.org/abs/2512.08186) variant splits that decision between two
systems running at different rates:

1. **System 2, a Qwen2.5-VL-7B planner.** It reads the instruction, up to 8 past frames and the current
   frame, and *generates text*: either discrete moves (`↑` forward, `←`/`→` turn, `↓` look down,
   `STOP`), or the pixel coordinates of the next waypoint in the image. For a pixel goal, a second
   prefill appends 4 learned query tokens to the conversation; their final hidden states become the
   **trajectory latents**, `[1, 4, 3584]`.
2. **System 1, a lightweight trajectory generator.** A DINOv2-S image encoder and a Q-Former compress
   two RGB frames into 32 memory tokens, which, together with the 4 projected trajectory latents, form
   36 conditioning tokens. A 12-layer NextDiT then denoises 32 candidate trajectories of 32 waypoints
   over 10 flow-matching Euler steps. Their mean is converted into local moves.

```
                                       ┌──► discrete moves (↑ ← → ↓ STOP)
  instruction ──┐                      │
                ├──► Qwen2.5-VL-7B ────┤
  RGB frames ───┘      (System 2)      └──► pixel goal ──► 4 trajectory latents ──┐
                                                                                  ▼
                 2 RGB frames ──► DINOv2 + Q-Former ──► 36 context ──► NextDiT ──► 32 candidate ──► local
                                    (System 1)           tokens      (10 Euler   trajectories     moves
                                                                      steps)
```

In this tutorial we build a PyTorch reference from the checkpoint, convert both systems to OpenVINO
Intermediate Representation (IR), run the converted pipeline on a sample observation, and check stage
by stage that the conversion preserved the model.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Prepare a sample observation](#Prepare-a-sample-observation)
- [Generate the PyTorch reference](#Generate-the-PyTorch-reference)
- [Convert the model to OpenVINO IR](#Convert-the-model-to-OpenVINO-IR)
    - [Why System 1 is re-implemented](#Why-System-1-is-re-implemented)
    - [Why the language model gets an extra output](#Why-the-language-model-gets-an-extra-output)
- [Select inference device](#Select-inference-device)
- [Load the OpenVINO pipeline](#Load-the-OpenVINO-pipeline)
- [Run System 2: plan from the instruction](#Run-System-2:-plan-from-the-instruction)
- [Run System 1: generate trajectories](#Run-System-1:-generate-trajectories)
- [Validate against the PyTorch reference](#Validate-against-the-PyTorch-reference)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/internvla-n1-dualvln/internvla-n1-dualvln.ipynb" />

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

> **Note:** `transformers` is pinned to 4.57.x. The checkpoint loader and the Qwen2.5-VL export were
> validated against that version, and installing `optimum-intel` unpinned pulls `transformers` 5.x.

In [ ]:
%pip install -q "torch>=2.6,<2.10" "torchvision" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "openvino>=2025.4" "optimum-intel[openvino]>=2.2" "transformers==4.57.*"
%pip install -q "diffusers==0.33.1" "timm>=1.0.20" "safetensors" "Pillow" "matplotlib" "ipywidgets"
%pip install -q "huggingface-hub<1.0,>=0.34.0"

In [ ]:
import shutil
from pathlib import Path

import requests

if not Path("notebook_utils.py").exists():
    # Prefer the repository copy; only standalone downloads of this notebook need the network.
    local_utils = Path("../../utils/notebook_utils.py")
    if local_utils.exists():
        shutil.copy(local_utils, "notebook_utils.py")
    else:
        r = requests.get(
            url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
        )
        open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("internvla-n1-dualvln.ipynb")

The notebook needs the [InternVLA-N1-DualVLN](https://huggingface.co/InternRobotics/InternVLA-N1-DualVLN)
checkpoint (about 16 GB). The cell below checks for it under `pretrained/` and downloads it from the
Hugging Face Hub if it is missing.

In [ ]:
import numpy as np
import openvino as ov

import internvla_helper as helper

# Everything the notebook writes stays next to the notebook.
CHECKPOINT_DIR = Path("pretrained/InternVLA-N1-DualVLN")
MODEL_DIR = Path("openvino_model")
REFERENCE_DIR = Path("golden_outputs")

if not (CHECKPOINT_DIR / "model.safetensors.index.json").exists():
    import download_checkpoints

    download_checkpoints.download_checkpoint(CHECKPOINT_DIR.parent)

for name in ["config.json", "model.safetensors.index.json", "generation_config.json", "preprocessor_config.json", "tokenizer_config.json"]:
    assert (CHECKPOINT_DIR / name).exists(), f"Missing {name} in {CHECKPOINT_DIR}"
print(f"Checkpoint ready in {CHECKPOINT_DIR}")

## Prepare a sample observation
[back to top ⬆️](#Table-of-contents:)

InternVLA-N1 was trained on Habitat renderings from the
[InternData-N1](https://huggingface.co/datasets/InternRobotics/InternData-N1) dataset, which is gated
and several terabytes in size. To keep this tutorial light, we use a deterministic stand-in observation
instead: two history frames plus the current frame, each a 384×384 RGB image drawn from a fixed seed,
together with an instruction from the R2R benchmark. The frames do not show a real scene, so the plan the
model produces is not a meaningful navigation decision, but every number below is reproducible and
directly comparable between PyTorch and OpenVINO. To try the model on your own data, replace `images`
with a list of PIL images, oldest first and current frame last.

The prompt follows the InternNav agent: the instruction, the history frames, and finally the current
frame.

In [ ]:
from PIL import Image

INSTRUCTION = (
    "Go straight to the hallway and then turn left. Go past the bed. " "Veer to the right and go through the white door. Stop when you're in the doorway."
)
SEED = 0

images = helper.make_fixed_observations(num_history=2, seed=SEED)
messages = helper.build_messages(INSTRUCTION, images)

print(" ".join("<image>" if part["type"] == "image" else part["text"] for part in messages[0]["content"]))
display(Image.fromarray(np.hstack([np.asarray(image) for image in images])))

## Generate the PyTorch reference
[back to top ⬆️](#Table-of-contents:)

Before converting anything we run the whole pipeline in PyTorch and keep every intermediate tensor:
the processor outputs, the generated token ids, the trajectory latents, the System 1 context, and the
input and output of each of the 10 denoising steps.

The reference is rebuilt from the checkpoint using only `transformers` and the modules in `internvla/`
(the same code that is converted). The upstream project produces these tensors by importing the full
InternNav tree, which pins an older `transformers` and pulls in Depth Anything and LongCLIP code that
inference never touches; rebuilding just the inference path keeps this notebook standalone and
compares like with like.

In the agent, System 1 only runs when System 2 answers with a pixel goal. Here we always compute the
trajectory latents and run System 1 on them, so every converted component is exercised on the same
sample, whichever branch System 2 picks. The two System 1 frames are both the current frame, and the
initial diffusion noise is drawn from a fixed seed.

This loads the 7B model in FP32 on CPU and takes several minutes.

In [ ]:
if (REFERENCE_DIR / "trajectories.npy").exists():
    print(f"Reference tensors already present in {REFERENCE_DIR} — skipping.")
else:
    helper.generate_reference(REFERENCE_DIR, CHECKPOINT_DIR, INSTRUCTION, images, seed=SEED)

reference = {p.stem: np.load(p) for p in sorted(REFERENCE_DIR.glob("*.npy"))}
for name in ["input_ids", "pixel_values", "generated_ids", "trajectory_latents", "s1_context", "trajectories"]:
    print(f"  {name:20s} {reference[name].shape}")

## Convert the model to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

The two systems are converted by different routes, because they are different kinds of model:

| Component | Route | Result |
|---|---|---|
| Qwen2.5-VL-7B (System 2) | [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) `export_from_model`, stateful | 4 `openvino_*_model.xml` IRs |
| System 1 context encoder: DINOv2-S, memory encoder, Q-Former, latent projector | `ov.convert_model` on a traced module | `s1_context_encoder.xml` |
| System 1 denoiser: NextDiT | `ov.convert_model` on a traced module | `s1_denoiser.xml` |

Qwen2.5-VL is already enabled in Optimum Intel, which splits it into vision embeddings, vision merger,
text embeddings and a stateful language model, and wires them together behind
`OVModelForVisualCausalLM`. All learned weights are saved in FP16, while model inputs and outputs
remain FP32.

`export.py` also checks its own work on CPU in FP32: each System 1 IR against its PyTorch module, the
32 tokens generated for a synthetic prompt against the PyTorch model token for token, and the
trajectory latents with a cosine similarity of at least 0.9999.

The conversion loads the model in FP32 and takes a while.

In [ ]:
import subprocess
import sys

if (MODEL_DIR / "export_manifest.json").exists():
    print(f"IRs already present in {MODEL_DIR} — skipping conversion.")
else:
    cmd = [sys.executable, "export.py", "--model-dir", str(CHECKPOINT_DIR), "--output-dir", str(MODEL_DIR)]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
for xml in sorted(MODEL_DIR.glob("*.xml")):
    mb = xml.with_suffix(".bin").stat().st_size / 2**20
    print(f"  {xml.name:48s} {mb:9.1f} MB")

## Select inference device
[back to top ⬆️](#Table-of-contents:)

Select the device from the dropdown list to run inference on. An Intel GPU runs the pipeline in FP16
and is much faster for the 7B planner; `CPU` runs it in FP32 and is useful as an exact-precision
reference.

In [ ]:
from notebook_utils import device_widget

gpus = [d for d in ov.Core().available_devices if d.startswith("GPU")]
device = device_widget(default=gpus[0] if gpus else "CPU", exclude=["NPU", "AUTO"])

device

## Load the OpenVINO pipeline
[back to top ⬆️](#Table-of-contents:)

`System2Runner` loads the four System 2 IRs through `OVModelForVisualCausalLM` and exposes
`generate` (text plan) and `generate_trajectory_latents` (the second prefill with the 4 learned queries).
`System1Runner` compiles the context encoder and the denoiser, and runs the flow-matching loop.

In [ ]:
from internvla.inference import System1Runner, System2Runner, parse_generated_text, trajectories_to_actions

precision = "f32" if device.value == "CPU" else "f16"
system2 = System2Runner(MODEL_DIR, device.value, precision)
system1 = System1Runner(MODEL_DIR, device.value, precision)

## Run System 2: plan from the instruction
[back to top ⬆️](#Table-of-contents:)

System 2 greedily generates a short answer. Digits in the answer mean a pixel goal `(x, y)` in the
current frame; otherwise the arrows are parsed as discrete moves.

In [ ]:
inputs = system2.preprocess_messages(messages, images)
generated_ids, generated_text, s2_seconds = system2.generate(inputs, max_new_tokens=128)
decision = parse_generated_text(generated_text)

print(f"prompt length    : {inputs['input_ids'].shape[1]} tokens ({len(images)} images)")
print(f"generated        : {generated_ids.shape[1] - inputs['input_ids'].shape[1]} tokens in {s2_seconds:.2f} s")
print(f"System 2 answer  : {generated_text!r}")
if decision["branch"] == "pixel_goal":
    print(f"decision         : pixel goal at (y, x) = {decision['pixel_goal_yx']}")
else:
    print(f"decision         : {[helper.ACTION_NAMES[a] for a in decision['actions']]}")

## Run System 1: generate trajectories
[back to top ⬆️](#Table-of-contents:)

Next we run the trajectory branch: a second System 2 prefill produces the trajectory latents from the
conversation above, and System 1 turns them, together with two RGB frames and a seeded noise sample,
into 32 candidate trajectories. As in the reference, both frames are the current frame. Each trajectory
is a sequence of 32 relative waypoints, which we integrate into a path seen from above, with the robot
at the origin facing up. The mean path is then followed with 25 cm steps and 15° turns to get the
first local moves.

In [ ]:
trajectory_latents, latent_seconds = system2.generate_trajectory_latents(generated_ids, inputs["pixel_values"], inputs["image_grid_thw"])
images_dp = helper.system1_images(images[-1], images[-1])
trajectories, s1_timings = system1.generate_trajectories(trajectory_latents, images_dp, helper.initial_noise(SEED))
local_actions = trajectories_to_actions(trajectories)[:4]

print(f"trajectory latents : {trajectory_latents.shape} in {latent_seconds:.2f} s")
print(
    f"trajectories       : {trajectories.shape} in {s1_timings['total_seconds']:.2f} s "
    f"(context {s1_timings['context_seconds'] * 1000:.0f} ms, 10 denoising steps {s1_timings['denoiser_seconds'] * 1000:.0f} ms)"
)
print(f"local moves        : {[helper.ACTION_NAMES[a] for a in local_actions]}")

In [ ]:
import matplotlib.pyplot as plt

paths = helper.trajectory_paths(trajectories)
mean_path = paths.mean(axis=0)

fig, (ax_image, ax_plan) = plt.subplots(1, 2, figsize=(11, 5))
ax_image.imshow(images[-1])
ax_image.set_title("current frame")
ax_image.axis("off")

# Robot frame: x forward, y left. Plot with forward up and right to the right.
for path in paths:
    ax_plan.plot(-path[:, 1], path[:, 0], color="tab:blue", alpha=0.25, linewidth=1)
ax_plan.plot(-mean_path[:, 1], mean_path[:, 0], color="tab:red", linewidth=2.5, label="mean trajectory")
ax_plan.scatter([0], [0], marker="^", s=150, color="black", zorder=3, label="robot")
ax_plan.set_xlabel("right (m)")
ax_plan.set_ylabel("forward (m)")
ax_plan.set_title("32 candidate trajectories (top view)")
ax_plan.axis("equal")
ax_plan.grid(alpha=0.3)
ax_plan.legend()
plt.tight_layout()
plt.show()

## Validate against the PyTorch reference
[back to top ⬆️](#Table-of-contents:)

Now we can check whether the conversion changed the model by comparing each stage against the tensors
captured from the PyTorch run. Each stage is fed the *reference* inputs where possible, so it is judged
on its own rather than on drift accumulated upstream:

* **Processor outputs** must match exactly: both sides use the same tokenizer and image processor.
* **Generated tokens are compared exactly, not by a similarity score.** Greedy decoding is a chain of
  argmax decisions, and a single different token can change the plan from "turn left" to a pixel goal,
  so the generated ids, the parsed branch and the moves must all be identical.
* **Trajectory latents** are produced from the reference token ids and compared by cosine similarity.
* **System 1 context** is produced from the reference latents and frames.
* **Each denoising step** is fed the reference noisy trajectories and context.
* **Final trajectories** come from the full OpenVINO chain and are gated on a relative L2 error below
  1%, and the local moves derived from them must match.

In [ ]:
cos_sim, rel_l2 = helper.cos_sim, helper.rel_l2
rows = []  # (stage, metric, value, gate, passed)

for name in ["input_ids", "attention_mask", "image_grid_thw", "pixel_values"]:
    exact = np.array_equal(inputs[name], reference[name])
    rows.append((f"processor {name}", "exact", float(exact), "==", exact))

tokens_exact = np.array_equal(generated_ids, reference["generated_ids"])
rows.append(("generated token ids", "exact", float(tokens_exact), "==", tokens_exact))

ref_continuation = reference["generated_ids"][0, reference["input_ids"].shape[1] :]
ref_decision = parse_generated_text(system2.tokenizer.decode(ref_continuation, skip_special_tokens=True))
rows.append(("System 2 decision", "exact", float(decision == ref_decision), "==", decision == ref_decision))

ov_latents, _ = system2.generate_trajectory_latents(reference["generated_ids"], reference["pixel_values"], reference["image_grid_thw"])
value = cos_sim(reference["trajectory_latents"], ov_latents)
rows.append(("trajectory latents", "cosine", value, 0.999, value >= 0.999))

ov_context = system1.context({"images": reference["images_dp"].transpose(0, 1, 4, 2, 3), "trajectory_latents": reference["trajectory_latents"]})["context"]
value = cos_sim(reference["s1_context"], ov_context)
rows.append(("System 1 context", "cosine", value, 0.999, value >= 0.999))

sigmas = helper.flow_match_sigmas()
for i, sigma in enumerate(sigmas[:-1]):
    step_inputs = helper.denoiser_step_inputs(reference[f"s1_trajectories_step{i}"], sigma, reference["s1_context"])
    prediction = system1.denoiser(step_inputs)["noise_prediction"]
    value = cos_sim(reference[f"s1_noise_prediction_step{i}"], prediction)
    rows.append((f"denoiser step {i} (t={step_inputs['timestep'][0]})", "cosine", value, 0.999, value >= 0.999))

value = rel_l2(reference["trajectories"], trajectories)
rows.append(("final trajectories", "rel. L2", value, 0.01, value <= 0.01))
ref_actions = trajectories_to_actions(reference["trajectories"])[:4]
rows.append(("local moves", "exact", float(local_actions == ref_actions), "==", local_actions == ref_actions))

print(f"{'stage':<32}{'metric':>9}{'value':>12}{'gate':>8}   result")
print("-" * 72)
for stage, metric, value, gate, passed in rows:
    gate_text = gate if isinstance(gate, str) else f"{gate:.3f}"
    print(f"{stage:<32}{metric:>9}{value:>12.6f}{gate_text:>8}   {'PASS' if passed else 'FAIL'}")
print("-" * 72)
print(f"reference answer: {system2.tokenizer.decode(ref_continuation, skip_special_tokens=True)!r}")